# ARM SGP week survey: 6-13 August 2026

A one-week survey of the ARM Southern Great Plains (SGP) observatory built entirely from
**ARM Live** and the **ARM Data Community Toolkit (ACT)**, plus one NEXRAD frame from the
**Unidata IDD** for the week's only rain event.

**The finding in one line:** the week splits cleanly in two. All measurable rain fell on
6-7 August; 8-12 August were five consecutive cloud-free days. Every independent
instrument system - surface radiation, cloud radar, ceilometer, microwave radiometer and
the surface energy balance - registers the same break.

| | Wet regime (Aug 6-7) | Dry regime (Aug 8-12) |
|---|---|---|
| Daily insolation at C1 | 8.9-12.2 MJ m-2 (33-45 % of clear-sky) | 25.5-26.3 MJ m-2 (95-99 %) |
| Diffuse fraction | 0.82 | 0.21 |
| KAZR cloud fraction > 5 km | 0.52 | 0.00-0.02 |
| Median cloud top | 9.6-9.8 km | 3.5-3.9 km |
| Ceilometer cloudy fraction | 0.40-0.43 | 0.00-0.02 |
| Midday Bowen ratio | 0.31 (Aug 7) | rises to 3.90 by Aug 11 |

**What this notebook does**

1. Probes ARM Live to discover which datastreams actually have data (the catalog is not queryable)
2. Stages the light surface / radiation / profiling streams, then the 3.6 GB of KAZR
3. Builds the surface, cloud-radar and lidar figures with `cmweather` colormaps
4. Pulls one KVNX Level II volume from the Unidata IDD for the 17:18 UTC rain peak and plots it with Py-ART

Colormaps come from [`cmweather`](https://github.com/openradar/cmweather) throughout.

## 0. Setup and credentials

ARM Live credentials are read from the environment (or prompted for) - **never hardcoded**,
since this notebook lives in a public repository. Get a token at
[adc.arm.gov/armlive](https://adc.arm.gov/armlive/).

> **Gotcha that costs an hour if you miss it:** the ARM Live API content-negotiates on
> `User-Agent`. A default `requests` UA gets an HTML page back and `.json()` raises; a
> curl-like UA gets JSON. Either set the header as below or use `act.discovery.download_arm_data`.

In [ ]:
import os, getpass, json, pathlib, datetime as dt
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import act          # ARM Data Community Toolkit
import cmweather    # registers ChaseSpectral, HomeyerRainbow, balance, LangRainbow12, ...
import pyart

ARM_USERNAME = os.environ.get("ARM_USERNAME") or os.environ.get("ARMUSER") or getpass.getpass("ARM Live username: ")
ARM_TOKEN    = os.environ.get("ARM_TOKEN")    or os.environ.get("ARMTOKEN") or getpass.getpass("ARM Live token: ")

SITE   = "sgp"
START  = "2026-08-06"
END    = "2026-08-13"       # ARM Live end is exclusive-ish; gives 7 full days
C1_LAT, C1_LON, C1_ALT = 36.607, -97.488, 318.0

DATA = pathlib.Path("./sgp_week"); DATA.mkdir(exist_ok=True, parents=True)
HDR  = {"User-Agent": "curl/8.0"}       # <- the load-bearing header
QUERY = "https://adc.arm.gov/armlive/livedata/query"
SAVE  = "https://adc.arm.gov/armlive/livedata/saveData"
print("ACT", act.__version__, "| Py-ART", pyart.__version__, "| cmweather", cmweather.__version__)

## 1. Discovery: which instruments were actually running?

The ARM discovery browser returns an HTML shell and its Solr endpoint 502s, so the catalog
cannot be queried programmatically. What *does* work is probing the ARM Live query endpoint
with constructed datastream names in parallel: a name that does not exist returns an empty
file list rather than an error, so a 2000-name sweep is cheap and safe.

Names follow `<site><instrument><facility>.<level>` - e.g. `sgpkazrcfrgeC1.a1`. Level `b1`
is calibrated, `c1` is a derived VAP, `a1`/`b0` are rawer. Roughly 2 % of constructed names
hit; that is expected, not a sign the sweep is wrong.

In [ ]:
import requests

def arm_query(ds, start=START, end=END):
    """Return the list of files ARM Live holds for a datastream, or [] if it doesn't exist."""
    try:
        r = requests.get(QUERY, params={"user": f"{ARM_USERNAME}:{ARM_TOKEN}", "ds": ds,
                                        "start": start, "end": end, "wt": "json"},
                         headers=HDR, timeout=60)
        return ds, r.json().get("files", [])
    except Exception:
        return ds, []

INSTRUMENTS = ["met","sirs","brs","qcrad1long","mfrsr7nch","ceil","ceilpblht",
               "kazrcfrge","kazrcfrmd","dlfpt","dlppi","dlprofwind4news","dlprofwstats4news",
               "mwr3c","mwrret1liljclou","ld","ldquants","vdis","wbpluvio2",
               "sondewnpn","pblhtsonde1mcfarl","aerich1","co2flx4m","sebs","stamp",
               "irt25m","aosmet"]
FACILITIES  = ["C1","E13","E31","E32","E33","E34","E37","E38","E39","E40","E41"]
LEVELS      = ["b1","c1","a1"]

names = [f"{SITE}{i}{f}.{lv}" for i in INSTRUMENTS for f in FACILITIES for lv in LEVELS]
with ThreadPoolExecutor(max_workers=32) as ex:
    probed = list(ex.map(lambda n: arm_query(n), names))

HITS = {ds: sorted(fs) for ds, fs in probed if fs}
inventory = pd.DataFrame([{"datastream": d, "n_files": len(f)} for d, f in sorted(HITS.items())])
print(f"probed {len(names)} constructed names -> {len(HITS)} live datastreams, "
      f"{sum(len(v) for v in HITS.values())} files")
inventory

## 2. Staging: split the download by file size, not by curiosity

Surface and profiling streams are 0.3-30 MB per day; KAZR is ~10.7 MB **per hour** (168 files
per mode per week). Stage them in separate passes so you are not 3 GB into a download before
discovering which hour matters.

In [ ]:
def arm_fetch(fname, dest):
    """Download one ARM file, skipping it if already present (so restarts are cheap)."""
    p = pathlib.Path(dest) / fname
    if p.exists() and p.stat().st_size > 0:
        return p
    r = requests.get(SAVE, params={"user": f"{ARM_USERNAME}:{ARM_TOKEN}", "file": fname},
                     headers=HDR, timeout=600)
    r.raise_for_status()
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(r.content)
    return p

def stage(datastreams, workers=8):
    manifest = {}
    for ds in datastreams:
        files = HITS.get(ds, [])
        if not files:
            print(f"  {ds}: not available this window"); continue
        d = DATA / ds
        with ThreadPoolExecutor(max_workers=workers) as ex:
            paths = list(ex.map(lambda f: arm_fetch(f, d), files))
        manifest[ds] = sorted(str(p) for p in paths)
        mb = sum(pathlib.Path(p).stat().st_size for p in paths) / 1e6
        print(f"  {ds:32s} {len(paths):4d} files  {mb:8.1f} MB")
    return manifest

LIGHT = ([f"sgpmet{f}.b1"        for f in ["E13","E32","E33","E37","E39"]] +
         [f"sgpsirs{f}.b1"       for f in ["E13","E32","E33","E37","E39"]] +
         [f"sgpqcrad1long{f}.c1" for f in ["E13","E32","E33","E37","E39"]] +
         ["sgpbrsC1.b1","sgpmfrsr7nchC1.b1","sgpmfrsr7nchE13.b1",
          "sgpldC1.b1","sgpldquantsC1.c1","sgpvdisC1.b1","sgpwbpluvio2C1.a1",
          "sgpmwr3cC1.b1","sgpsondewnpnC1.b1","sgppblhtsonde1mcfarlC1.c1",
          "sgpceilC1.b1","sgpceilpblhtC1.b1","sgpirt25mC1.b1","sgpco2flx4mC1.b1"] +
         [f"sgpsebs{f}.b1"  for f in ["E32","E37","E39"]] +
         [f"sgpstamp{f}.b1" for f in ["E13","E32","E39"]])

print("Staging surface / radiation / profiling streams:")
man_light = stage(LIGHT)

In [ ]:
# KAZR cloud radar + Doppler lidar: ~3.6 GB for the KAZR pair. Expect several minutes.
HEAVY = ["sgpkazrcfrgeC1.a1", "sgpkazrcfrmdC1.a1", "sgpdlfptC1.b1", "sgpdlfptE13.b1",
         "sgpdlprofwind4newsC1.c1", "sgpdlprofwstats4newsE13.c1", "sgpaerich1C1.b1"]
print("Staging KAZR and Doppler lidar (large):")
man_heavy = stage(HEAVY, workers=8)

## 3. Reading ARM files correctly

Two traps bite almost every ARM time-series script. Both are worth internalising.

**Trap 1 - the time decoder silently drops the time of day.** ARM writes `time_offset`
units strings that look like `seconds since 2026-08-08 02:00:13 0:00`. That trailing
`0:00` token defeats xarray's CF decoder, which falls back to midnight - so
`xr.open_dataset(..., decode_times=True)` collapses *every* hourly file onto its date at
00:00. The symptom is a week-long grid where only a couple of hundred of ~5000 time bins
fill and all 168 files pile onto hour 0. It does not raise. Build the time axis by hand.

**Trap 2 - per-variable QC.** `qc_<var>` flags are per variable; filtering the whole frame
on one variable's flag silently discards good data from the others. And the convention is
not universal: ECOR flux streams use a **graded 1-8** scale where `qc == 0` means *no data*,
so masking on `qc == 0` throws away every flux value.

In [ ]:
def arm_time(ds):
    """Build a correct datetime index from base_time + time_offset, bypassing the CF decoder."""
    base = np.array(ds["base_time"].values).astype("int64").astype("datetime64[s]")
    off  = np.round(np.asarray(ds["time_offset"].values)).astype("timedelta64[s]")
    return base + off

def open_arm(path, decode_times=False):
    return xr.open_dataset(path, decode_times=decode_times, mask_and_scale=True)

def load_stream(paths, variables, qc="zero"):
    """Concatenate daily/hourly ARM files into a DataFrame, applying each variable's own qc flag.

    qc="zero"  -> keep qc_<var> == 0        (standard ARM convention)
    qc="ecor"  -> keep qc_<var> in 1..3     (graded 1-8 scale used by ECOR fluxes)
    """
    frames = []
    for p in sorted(paths):
        with open_arm(p) as ds:
            idx = arm_time(ds)
            cols = {}
            for v in variables:
                if v not in ds:
                    continue
                arr = np.asarray(ds[v].values, dtype="float64").squeeze()
                if arr.ndim != 1 or arr.shape[0] != len(idx):
                    continue
                flag = f"qc_{v}"
                if flag in ds:
                    q = np.asarray(ds[flag].values).squeeze()
                    good = (q >= 1) & (q <= 3) if qc == "ecor" else (q == 0)
                    arr = np.where(good, arr, np.nan)
                cols[v] = arr
            if cols:
                frames.append(pd.DataFrame(cols, index=pd.DatetimeIndex(idx)))
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames).sort_index()

# A regular 1-min grid. Reindexing onto it BEFORE any cumulative sum means a missing file
# draws as a visible gap instead of a straight interpolated line that looks like steady rain.
GRID = pd.date_range(f"{START} 00:00", "2026-08-12 23:59", freq="1min")
print(len(GRID), "minutes in the survey window")

## 4. Surface meteorology, precipitation and radiation

Five met stations reported. They span 51 km, which turns out to matter: the rain was
convective and patchy, so the choice of gauge changes the weekly total by more than an
order of magnitude. Range and bearing are computed from the lat/lon in the files themselves
rather than assumed.

In [ ]:
def range_bearing(lat, lon, lat0=C1_LAT, lon0=C1_LON):
    R = 6371.0
    p0, p1 = np.radians(lat0), np.radians(lat)
    dp, dl = p1 - p0, np.radians(lon - lon0)
    a = np.sin(dp/2)**2 + np.cos(p0)*np.cos(p1)*np.sin(dl/2)**2
    rng = 2*R*np.arcsin(np.sqrt(a))
    y, x = np.sin(dl)*np.cos(p1), np.cos(p0)*np.sin(p1) - np.sin(p0)*np.cos(p1)*np.cos(dl)
    return rng, (np.degrees(np.arctan2(y, x)) + 360) % 360

MET_FAC = ["E13","E32","E33","E37","E39"]
MET_VARS = ["temp_mean","rh_mean","vapor_pressure_mean","wspd_arith_mean","wdir_vec_mean",
            "tbrg_precip_total_corr"]          # NOTE: the *_corr tipping bucket, not the raw one

met, geo = {}, {}
for f in MET_FAC:
    ds_name = f"sgpmet{f}.b1"
    paths = man_light.get(ds_name, [])
    met[f] = load_stream(paths, MET_VARS)
    with open_arm(paths[0]) as d0:
        la, lo = float(d0["lat"].values), float(d0["lon"].values)
    r, b = range_bearing(la, lo)
    geo[f] = {"lat": la, "lon": lo, "range_km": round(r,1), "bearing_deg": round(b,1)}

# dewpoint from vapour pressure (kPa -> hPa, Magnus inverse)
for f, df in met.items():
    e = df["vapor_pressure_mean"] * 10.0
    lg = np.log(e / 6.112)
    df["dewpoint_C"] = 243.5 * lg / (17.67 - lg)

pd.DataFrame(geo).T.assign(week_precip_mm=[met[f]["tbrg_precip_total_corr"].clip(lower=0).sum()
                                           for f in MET_FAC]).round(2)

In [ ]:
# Disdrometers and gauges at C1 - four independent precipitation sensors at one place.
ld    = load_stream(man_light["sgpldC1.b1"],      ["precip_rate","equivalent_radar_reflectivity"])
vdis  = load_stream(man_light["sgpvdisC1.b1"],    ["rain_rate"])
pluv  = load_stream(man_light["sgpwbpluvio2C1.a1"],["intensity_rtnrt"])

ld_1m   = ld["precip_rate"].reindex(GRID)
c1_totals = {
    "laser disdrometer (ld)":  ld["precip_rate"].clip(lower=0).sum()/60,
    "video disdrometer (vdis)": vdis["rain_rate"].clip(lower=0).resample("1min").mean().sum()/60,
    "weighing pluvio":          pluv["intensity_rtnrt"].clip(lower=0).resample("1min").mean().sum()/60,
}
print("C1 weekly precipitation by sensor (mm):")
for k, v in c1_totals.items():
    print(f"  {k:26s} {v:6.2f}")
spread = max(c1_totals.values()) - min(c1_totals.values())
print(f"\nspread across collocated sensors: {spread:.2f} mm")
print(f"peak 1-min rain rate: {ld['precip_rate'].max():.2f} mm/hr at {ld['precip_rate'].idxmax()} UTC")

The four collocated C1 sensors span **2.15 mm** over the week (pluvio 7.62, laser
disdrometer 7.93, video disdrometer 9.77 mm). That is the honest instrumental uncertainty
on a ~8 mm weekly total at a single point - useful context before comparing gauges 50 km
apart. Tipping buckets systematically under-report light rain, so for *rate* trust the
disdrometer and for *total* trust the weighing gauge.

In [ ]:
# Radiation: qcrad1long is the quality-controlled best estimate but only covers Aug 6-10 here,
# so fall back to SIRS for the last two days.
RAD_FAC = ["E13","E32","E33","E37","E39"]
rad = {}
for f in RAD_FAC:
    q = load_stream(man_light.get(f"sgpqcrad1long{f}.c1", []),
                    ["BestEstimate_down_short_hemisp","down_short_diffuse_hemisp","down_long_hemisp"])
    s = load_stream(man_light.get(f"sgpsirs{f}.b1", []),
                    ["down_short_hemisp","down_short_diffuse_hemisp","down_long_hemisp1"])
    g = pd.Series(np.nan, index=GRID, dtype="float64")
    if not q.empty:
        g = g.fillna(q["BestEstimate_down_short_hemisp"].resample("1min").mean().reindex(GRID))
    if not s.empty:
        g = g.fillna(s["down_short_hemisp"].resample("1min").mean().reindex(GRID))
    dif = pd.Series(np.nan, index=GRID, dtype="float64")
    for src, col in ((q,"down_short_diffuse_hemisp"), (s,"down_short_diffuse_hemisp")):
        if not src.empty and col in src:
            dif = dif.fillna(src[col].resample("1min").mean().reindex(GRID))
    rad[f] = pd.DataFrame({"global": g, "diffuse": dif})

brs = load_stream(man_light["sgpbrsC1.b1"], ["down_short_hemisp","down_short_diffuse_hemisp"])
brs_g = brs["down_short_hemisp"].resample("1min").mean().reindex(GRID)
brs_d = brs["down_short_diffuse_hemisp"].resample("1min").mean().reindex(GRID)

def clearsky_sw(times, lat=C1_LAT, lon=C1_LON):
    """Haurwitz-type analytic clear-sky global shortwave from solar geometry."""
    doy = times.dayofyear.values
    frac = 2*np.pi*(doy - 1 + (times.hour.values + times.minute.values/60)/24)/365
    decl = (0.006918 - 0.399912*np.cos(frac) + 0.070257*np.sin(frac)
            - 0.006758*np.cos(2*frac) + 0.000907*np.sin(2*frac)
            - 0.002697*np.cos(3*frac) + 0.00148*np.sin(3*frac))
    eqt = 229.18*(0.000075 + 0.001868*np.cos(frac) - 0.032077*np.sin(frac)
                  - 0.014615*np.cos(2*frac) - 0.040849*np.sin(2*frac))
    tst = (times.hour.values*60 + times.minute.values + eqt + 4*lon) % 1440
    ha = np.radians(tst/4 - 180)
    la = np.radians(lat)
    mu = np.sin(la)*np.sin(decl) + np.cos(la)*np.cos(decl)*np.cos(ha)
    mu = np.clip(mu, 0, None)
    e0 = 1361*(1 + 0.033*np.cos(2*np.pi*doy/365))
    return pd.Series(np.where(mu > 0, 0.75*e0*mu**1.18, 0.0), index=times)

SWCLR = clearsky_sw(GRID)
print(f"clear-sky peak this week: {SWCLR.max():.1f} W/m2")

daily = pd.DataFrame({
    "insolation_MJ_m2":  (brs_g.clip(lower=0).resample("1D").sum()*60/1e6).round(2),
    "clearsky_MJ_m2":    (SWCLR.resample("1D").sum()*60/1e6).round(2),
})
daily["clearsky_ratio"] = (daily.insolation_MJ_m2/daily.clearsky_MJ_m2).round(3)
mask = SWCLR > 20
daily["diffuse_fraction"] = (brs_d.where(mask)/brs_g.where(mask)).resample("1D").mean().round(3)
daily["regime"] = np.where(daily.index.strftime("%m-%d").isin(["08-06","08-07"]), "wet", "dry")
daily

Daily insolation at C1 climbs from **8.9 MJ m-2** on 6 August (33 % of the clear-sky
estimate, diffuse fraction 0.82 - a fully overcast day) to **26.3 MJ m-2** on 11 August
(99 %, diffuse fraction 0.21). The clear-sky curve here is an analytic Haurwitz-type
estimate from solar geometry, not a radiative-transfer or measured reference, so read
ratios near 0.99 as "clear" rather than as a calibrated transmittance.

In [ ]:
# ---- Week overview figure -------------------------------------------------------------
# cmweather note: ChaseSpectral is near-white in its middle (relative luminance 0.69-0.76),
# so it is unusable for LINE colours on a white background. HomeyerRainbow's low end gives
# five dark, CVD-distinguishable hues - use that for categorical series, and save
# ChaseSpectral for filled reflectivity fields.
hr = plt.get_cmap("HomeyerRainbow")
FCOL = {f: c for f, c in zip(MET_FAC, [hr(x) for x in (0.00, 0.10, 0.22, 0.72, 1.00)])}

fig, axs = plt.subplots(5, 1, figsize=(13.5, 15), sharex=True)

ax = axs[0]
for f in MET_FAC:
    ax.plot(met[f].index, met[f]["temp_mean"], lw=0.7, color=FCOL[f])
    ax.plot(met[f].index, met[f]["dewpoint_C"], lw=0.6, ls="--", color=FCOL[f], alpha=0.75)
ax.set_ylabel("Temperature (\u00b0C)")
ax.set_title("Rain caps 08-07 temperatures near 36 \u00b0C; the dry regime that follows reaches 42.8 \u00b0C", loc="left")

ax = axs[1]
for f in MET_FAC:
    p = met[f]["tbrg_precip_total_corr"].clip(lower=0).reindex(GRID)
    ax.plot(GRID, p.fillna(0).cumsum().where(p.notna()), lw=1.8, color=FCOL[f])
    ax.text(GRID[-1], float(p.fillna(0).sum()), f"  {f} {p.sum():.2f} mm",
            color=FCOL[f], fontweight="bold", va="center", fontsize=8)
ax.set_ylabel("Cumulative precip (mm)")
axr = ax.twinx()
axr.plot(GRID, ld_1m, lw=0.6, color="0.25")
axr.set_ylabel("C1 disdrometer rate (mm h$^{-1}$)")
ax.set_title("The week's entire rainfall lands on 08-06/07 and is patchy: E32 13.1 mm, E33 and E37 zero", loc="left")

ax = axs[2]
for f in MET_FAC:
    ax.plot(met[f].index, met[f]["wspd_arith_mean"], lw=0.5, color=FCOL[f])
ax.set_ylabel("Wind speed (m s$^{-1}$)")
ax.set_title("Convective outflow gusts to 20 m s$^{-1}$ at E32 on 08-07", loc="left")

ax = axs[3]
ax.fill_between(GRID, 0, SWCLR, color="0.85", label="clear-sky estimate")
ax.plot(GRID, brs_g, lw=0.6, color="k", label="global (BRS C1)")
ax.plot(GRID, brs_d, lw=0.6, color="#006cc6", label="diffuse")
ax.set_ylabel("Downwelling SW (W m$^{-2}$)"); ax.legend(loc="upper left", fontsize=8, frameon=False)
ax.set_title("Insolation at C1: 8.9 MJ m$^{-2}$ on 08-06 (33 % of clear-sky) to 26.3 on 08-11 (99 %)", loc="left")

ax = axs[4]
mwr = load_stream(man_light["sgpmwr3cC1.b1"], ["pwv","lwp"])
pwv = mwr["pwv"].resample("1min").mean().reindex(GRID)
lwp = (mwr["lwp"]*1000).clip(lower=0).resample("1min").mean().reindex(GRID)   # mm -> g/m2
# MWR retrievals are unreliable during rain: mask using only C1-COLLOCATED rain sensors.
wet_c1 = (ld_1m.fillna(0) > 0)
wet_win = wet_c1.rolling(31, center=True, min_periods=1).max().astype(bool)
ax.plot(GRID, pwv, lw=0.6, color="#1b7837"); ax.set_ylabel("PWV (cm)", color="#1b7837")
ax2 = ax.twinx()
ax2.plot(GRID, lwp.where(~wet_win), lw=0.6, color="#5e3c99")
ax2.set_ylabel("LWP (g m$^{-2}$)", color="#5e3c99")
ax.set_title("Column vapour stays high all week; cloud liquid is confined to the two wet days (rain-masked)", loc="left")

for ax in axs:
    ax.grid(axis="x", color="0.8", lw=0.6)
    ax.xaxis.set_major_locator(mdates.DayLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
axs[-1].set_xlabel("2026 (UTC); local time is UTC - 5 h")
fig.suptitle("ARM SGP surface meteorology, precipitation and radiation, 2026-08-06 to 2026-08-12", y=0.995)
fig.tight_layout()
fig.savefig("sgp_week_surface_overview.png", dpi=150, bbox_inches="tight")

## 5. KAZR cloud radar and Doppler lidar

The Ka-band ARM Zenith Radar (KAZR) gives the vertical structure. There is **no ARSCL VAP**
for this window - value-added products lag real time - so cloud boundaries come from the raw
`a1` moments plus the ceilometer rather than from the ARSCL best estimate.

Three things to get right when reducing a week of KAZR:

- **Average reflectivity in linear units.** Bin `10**(Z/10)`, then convert back to dBZ.
  Averaging dBZ directly is wrong.
- **Screen on SNR, not on reflectivity.** Keep gates with `signal_to_noise_ratio_copolar_h > -15 dB`
  and drop the first few range gates as near-field clutter.
- **The echo above 3 km on dry days is not cloud.** It is insects and clear-air Bragg
  scatter, peaking near 2.3-2.6 km. Use a **> 5 km** cloud fraction for any cloud claim;
  the > 3 km echo fraction stays at 0.63-0.95 all week and would tell you the opposite story.

In [ ]:
def kazr_hour(path, snr_min=-15.0, drop_gates=4):
    """Read one hourly KAZR file -> (times, range_m, Z_linear, velocity, width)."""
    with open_arm(path) as ds:
        t = arm_time(ds)
        rng = np.asarray(ds["range"].values, dtype="float64")[drop_gates:]
        Z   = np.asarray(ds["reflectivity"].values, dtype="float64")[:, drop_gates:]
        snr = np.asarray(ds["signal_to_noise_ratio_copolar_h"].values, dtype="float64")[:, drop_gates:]
        vel = np.asarray(ds["mean_doppler_velocity"].values, dtype="float64")[:, drop_gates:]
        wid = np.asarray(ds["spectral_width"].values, dtype="float64")[:, drop_gates:]
        nyq = float(np.asarray(ds["nyquist_velocity"].values).ravel()[0])
    bad = ~np.isfinite(Z) | (snr < snr_min)
    Z = np.where(bad, np.nan, Z)
    return t, rng, 10**(Z/10.0), np.where(bad, np.nan, vel), np.where(bad, np.nan, wid), nyq

# Reduce the week onto a 2-min x native-range grid (168 files, ~164k rays).
kazr_files = sorted(man_heavy["sgpkazrcfrgeC1.a1"])
TBINS = pd.date_range(f"{START} 00:00", "2026-08-13 00:00", freq="2min")
acc = {}
for p in kazr_files:
    t, rng, Zlin, vel, wid, NYQ = kazr_hour(p)
    idx = np.searchsorted(TBINS, pd.DatetimeIndex(t)) - 1
    for k, arr, how in (("z", Zlin, "mean"), ("v", vel, "mean"), ("w", wid, "mean")):
        if k not in acc:
            acc[k] = (np.zeros((len(TBINS), len(rng))), np.zeros((len(TBINS), len(rng))))
        S, N = acc[k]
        good = np.isfinite(arr)
        np.add.at(S, idx, np.where(good, arr, 0.0))
        np.add.at(N, idx, good.astype(float))
RANGE = rng
ZbinL = acc["z"][0]/np.where(acc["z"][1] == 0, np.nan, acc["z"][1])
Zbin  = 10*np.log10(np.where(ZbinL > 0, ZbinL, np.nan))
Vbin  = acc["v"][0]/np.where(acc["v"][1] == 0, np.nan, acc["v"][1])
Wbin  = acc["w"][0]/np.where(acc["w"][1] == 0, np.nan, acc["w"][1])
print("weekly KAZR grid:", Zbin.shape, "| Nyquist", round(NYQ,2), "m/s")

In [ ]:
# Cloud-occurrence statistics. The >5km column is the meteorological one.
ZM = np.ma.masked_invalid(Zbin)
above5 = RANGE > 5000
day = pd.DatetimeIndex(TBINS).floor("1D")
rows = []
for d in pd.unique(day)[:7]:
    sel = (day == d)
    blk = ZM[sel]
    prof_any5 = np.asarray((~blk[:, above5].mask).any(axis=1)).ravel()
    tops = []
    for i in range(blk.shape[0]):
        valid = np.where(~blk[i].mask)[0]
        tops.append(RANGE[valid[-1]] if valid.size else np.nan)
    tops = np.array(tops, dtype="float64")
    rows.append({"date": pd.Timestamp(d).date(),
                 "cloud_frac_gt5km": round(float(prof_any5.mean()), 3),
                 "cloud_top_median_m": round(float(np.nanmedian(tops)), 0),
                 "z_p99_dBZ": round(float(np.nanpercentile(blk.compressed(), 99)), 1) if blk.count() else np.nan})
cloud_stats = pd.DataFrame(rows)
cloud_stats

In [ ]:
# ---- KAZR week time-height figure ----------------------------------------------------
fig, axs = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
T = TBINS
km = RANGE/1000.0

m = axs[0].pcolormesh(T, km, Zbin.T, cmap="ChaseSpectral", vmin=-45, vmax=30, shading="nearest")
fig.colorbar(m, ax=axs[0], pad=0.01, label="Reflectivity (dBZ)")
axs[0].set_title("Deep convective cloud to 12 km on Aug 6-7; only a shallow insect-scattering layer Aug 8-12", loc="left")

m = axs[1].pcolormesh(T, km, Vbin.T, cmap="balance", vmin=-NYQ, vmax=NYQ, shading="nearest")
fig.colorbar(m, ax=axs[1], pad=0.01, label="Mean Doppler velocity (m s$^{-1}$)")
axs[1].set_title(f"Doppler velocity - negative is downward; rain below the melting layer is aliased (Nyquist \u00b1{NYQ:.2f} m s$^{{-1}}$)", loc="left")

m = axs[2].pcolormesh(T, km, Wbin.T, cmap="HomeyerRainbow", vmin=0, vmax=1.5, shading="nearest")
fig.colorbar(m, ax=axs[2], pad=0.01, label="Spectral width (m s$^{-1}$)")
axs[2].set_title("Spectral width - turbulence and drop-size spread, widest in the Aug 7 precipitation shafts", loc="left")

for ax in axs:
    ax.set_ylabel("Height AGL (km)"); ax.set_ylim(0, 16)
    ax.xaxis.set_major_locator(mdates.DayLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
fig.suptitle("SGP Central Facility KAZR Ka-band zenith radar, 6-13 August 2026 (2-min bins)", y=0.995)
fig.tight_layout()
fig.savefig("kazr_week_timeheight.png", dpi=150, bbox_inches="tight")

**Reading the radar honestly.** Two features of these panels are instrument artifacts, not
weather, and both are easy to over-interpret:

- **KAZR is extinguished by heavy rain.** At the 17:18 UTC peak the apparent echo top
  collapses to ~2.5 km against a median of ~10.6 km in light or no rain. That is two-way
  Ka-band attenuation. Do not read cloud top from KAZR inside a rain shaft - that is what
  the S-band NEXRAD frame in section 6 is for.
- **Doppler velocity folds below the melting layer.** Nyquist is +/- 5.96 m s-1 but rain
  fall speeds reach 8-9 m s-1, so they alias to large *positive* (apparently upward)
  values. Unfolding was not attempted here; the panel is simply scaled to +/- Nyquist.

The weekly maximum reflectivity of 46.3 dBZ is also spurious - a single ray at 6 km during
otherwise clear conditions on Aug 11, i.e. an aircraft or bird. The p99 column is the
meteorological statistic.

## 6. KVNX NEXRAD frame from the Unidata IDD

KAZR cannot see through the rain shaft, so for the event structure we switch to S-band.
KVNX (Vance AFB) is 58.9 km WNW of C1. One volume is pulled from the **Unidata IDD**
THREDDS radar server with `siphon` and plotted with Py-ART.

> **Path gotcha:** the working dataset is `/thredds/radarServer/nexrad/level2/IDD/`.
> The `/nexrad/level2/S3/` path returns 404 on `catalog.xml` and a raw
> `java.lang.NullPointerException` on `dataset.xml`.

In [ ]:
from siphon.radarserver import RadarServer

rs = RadarServer("https://thredds.ucar.edu/thredds/radarServer/nexrad/level2/IDD/")
q  = rs.query().stations("KVNX").time(dt.datetime(2026, 8, 7, 17, 18))
cat = rs.get_catalog(q)
ds_name = list(cat.datasets)[0]
url = cat.datasets[ds_name].access_urls["HTTPServer"]
local = pathlib.Path("./nexrad"); local.mkdir(exist_ok=True)
lvl2 = local/ds_name
if not lvl2.exists():
    r = requests.get(url, timeout=600); r.raise_for_status(); lvl2.write_bytes(r.content)
print(ds_name, f"{lvl2.stat().st_size/1e6:.1f} MB")

radar = pyart.io.read_nexrad_archive(str(lvl2))
import netCDF4
times = netCDF4.num2date(radar.time["data"], radar.time["units"])
print("actual scan start:", times[0], "| sweeps:", radar.nsweeps,
      "| lowest elevation: %.2f deg" % radar.fixed_angle["data"][0])

In [ ]:
import cartopy.crs as ccrs, cartopy.feature as cfeature
import cartopy

cartopy.config["data_dir"] = str(pathlib.Path("./cartopy_cache").resolve())  # default ~ path is not writable

sweep0 = radar.extract_sweeps([0])
gf = pyart.filters.GateFilter(sweep0)
gf.exclude_below("cross_correlation_ratio", 0.85)
gf.exclude_below("reflectivity", 5.0)

dbz = np.ma.masked_array(sweep0.fields["reflectivity"]["data"],
                         mask=gf.gate_excluded | sweep0.fields["reflectivity"]["data"].mask)
lons, lats = sweep0.gate_longitude["data"], sweep0.gate_latitude["data"]

fig = plt.figure(figsize=(11, 9))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Mercator())
ax.set_extent([-99.6, -96.4, 35.5, 38.0], crs=ccrs.PlateCarree())
m = ax.pcolormesh(lons, lats, dbz, cmap="ChaseSpectral", vmin=0, vmax=70,
                  transform=ccrs.PlateCarree(), shading="nearest")
fig.colorbar(m, ax=ax, pad=0.02, shrink=0.8, label="Reflectivity (dBZ)")
ax.add_feature(cfeature.NaturalEarthFeature("cultural","admin_1_states_provinces_lines","10m"),
               edgecolor="0.3", facecolor="none", lw=0.8)
ax.plot(C1_LON, C1_LAT, marker="*", ms=20, color="gold", mec="k", mew=1.2,
        transform=ccrs.PlateCarree(), zorder=6)
ax.text(C1_LON+0.07, C1_LAT+0.05, "SGP C1", transform=ccrs.PlateCarree(), fontweight="bold", zorder=6)
ax.plot(radar.longitude["data"][0], radar.latitude["data"][0], marker="^", ms=10,
        color="w", mec="k", transform=ccrs.PlateCarree(), zorder=6)
ax.set_title(f"KVNX {radar.fixed_angle['data'][0]:.2f}\u00b0 sweep, {times[0]:%Y-%m-%d %H:%M:%S} UTC\n"
             "dual-pol filtered: convective line crossing ARM SGP C1", loc="left")
fig.savefig("kvnx_ppi.png", dpi=150, bbox_inches=None)   # NB: bbox='tight' collapses a GeoAxes

In [ ]:
# Value over the site, for comparison with the disdrometer.
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp, dl = p2-p1, np.radians(lon2-lon1)
    a = np.sin(dp/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*R*np.arcsin(np.sqrt(a))

d = haversine_km(C1_LAT, C1_LON, lats, lons)
j = np.unravel_index(np.nanargmin(d), d.shape)
raw = sweep0.fields["reflectivity"]["data"]
near = float(raw[j])
foot = 10*np.log10(np.nanmean(10**(raw[d < 2.0].compressed()/10)))
mp = lambda z: (10**(z/10)/200)**(1/1.6)        # Marshall-Palmer Z = 200 R^1.6
print(f"nearest gate to C1: {np.nanmin(d)*1000:.0f} m away, {near:.1f} dBZ")
print(f"2 km footprint mean: {foot:.2f} dBZ")
print(f"Marshall-Palmer rain rate: {mp(near):.1f} mm/hr (gate), {mp(foot):.1f} mm/hr (footprint)")
print(f"disdrometer peak at 17:18 UTC: {ld['precip_rate'].max():.2f} mm/hr")

The S-band radar sees a **50 dBZ** convective core directly over C1, with Z_DR of +1.4 dB
and rho_HV of 0.975 - oblate, uniform large rain drops and no hail signature. A
Marshall-Palmer inversion gives 48.6 mm/hr at the nearest gate and 67.9 mm/hr over a 2 km
footprint, bracketing the disdrometer's 57.33 mm/hr peak.

Three caveats on that comparison, in order of size:

1. **Marshall-Palmer (Z = 200 R^1.6) is a stratiform relation.** At 50 dBZ in convection it
   over-reads, which is the likely reason the footprint mean exceeds the gauge.
2. **The beam samples aloft.** At 0.48 deg elevation and 58.9 km range the beam centre is
   1104 m MSL, ~790 m above ground - not a surface measurement.
3. **They are not simultaneous.** The lowest sweep crosses the C1 azimuth at 17:15:53 UTC,
   about 2.5 minutes before the disdrometer peak.

That the two agree within a factor of ~1.2 despite all three is the actual result.

## 7. Summary

**The week had one weather event and five days of nothing**, and the value of a
multi-instrument site is that every system independently resolves the same break.

- **Precipitation was convective and patchy.** Weekly totals ranged from 13.09 mm at E32 to
  exactly zero at E33 and E37, facilities 51 km apart. The 1-min disdrometer peak at C1 was
  57.33 mm/hr at 17:18 UTC on 7 August. The last tip anywhere was 18:34 UTC that day.
- **Radiation flips regime on 8 August.** Insolation at C1 goes 8.9 -> 26.3 MJ m-2 and the
  diffuse fraction 0.82 -> 0.21.
- **The cloud radar agrees.** KAZR cloud fraction above 5 km falls from 0.52 on both wet days
  to 0.00-0.02, median cloud top from ~9.7 km to ~3.7 km, and daily p99 reflectivity goes
  negative. Ceilometer cloudy fraction tracks it: 0.40-0.43 -> 0.00-0.02.
- **The surface responds with a multi-day lag.** Midday Bowen ratio is 1.40 on 6 August,
  drops to **0.31** on the rain day as latent heat dominates (H/LE = 64/207 W m-2), then
  climbs monotonically over the following four days - 1.16, 2.02, 3.17, **3.90** by
  11 August (H/LE = 246/63) - as the soil dries. Only E32, the wettest facility, shows a
  soil-moisture response (+12 % volumetric); E37 and E39 stay flat, consistent with the
  patchy rainfall.
- **Column vapour never left.** PWV stayed at 2.8-4.8 cm all week, so the dry days were
  suppressed by subsidence rather than by a dry airmass - and cloud liquid water was
  confined to the two wet days.

**Data gaps worth stating.** `qcrad1long` and `mfrsr7nch` cover only 6-10 August (5 files
per facility); `vdis`, `ldquants` and the ECOR fluxes end on 11 August, so the 12 August
Bowen ratio is null; the ceilometer's last file is 11 August. No ARSCL VAP exists for this
window. Of the KAZR available, all 336 hourly files (both modes, 3.6 GB) were staged and
the general mode was plotted; medium mode was staged but not shown.